# 04 · Scoring a CSV and exporting the queue

I wanted to use the trained model on a file without fraud labels. This notebook scores either the original held-out window or a compatible custom CSV. Each exported decision keeps its one-based source record so I can find that payment again.

In [11]:
from pathlib import Path
from datetime import datetime, timezone
from tempfile import TemporaryDirectory
from uuid import uuid4
import csv

import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display
from google.colab import drive

drive.mount("/content/drive")
project = Path("/content/drive/MyDrive/Colab Notebooks/Project 01: Intelligent Financial Fraud Detection")
run_id = "20261121T162317186996Z"
source = "Custom CSV" # @param ["Held-out replay", "Custom CSV"]
custom_csv = "/content/drive/MyDrive/Colab Notebooks/Project 01: Intelligent Financial Fraud Detection/Dataset/creditcard.csv" # @param {type:"string"}
use_saved_threshold = True # @param {type:"boolean"}
custom_threshold = 0.999 # @param {type:"number"}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Loading the checkpoint

I use the same features, weights, scaling and threshold as the reference run. The custom cutoff is useful when I want to replay a different review policy.

In [12]:
saved = torch.load(project / "Artifacts" / f"ann_{run_id}.pt", map_location="cpu", weights_only=True)
features = saved["features"]
model = nn.Sequential(nn.Linear(len(features), 64), nn.ReLU(), nn.Dropout(0.2),
                      nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
model.load_state_dict(saved["state_dict"])
model.eval()
cutoff = float(saved["threshold"] if use_saved_threshold else custom_threshold)
if not 0 <= cutoff <= 1:
    raise ValueError("Choose a threshold from 0 to 1.")
print(f"Operating threshold = {cutoff:.10f}")

Operating threshold = 0.9998766184


## Checking the incoming file

A batch can have its columns in a different order. I select the training features by name and check for missing inputs, duplicate headers and nonfinite values before scoring. Held-out replay uses the original chronological split.

In [13]:
def read_batch(path, replay):
    with path.open(encoding="utf-8-sig", newline="") as stream:
        header = next(csv.reader(stream), [])
    if len(header) != len(set(header)):
        raise ValueError("CSV contains duplicate column names.")
    missing = sorted(set(features) - set(header))
    if missing:
        raise ValueError(f"Missing required features: {missing}")
    # Read inputs only. Class and any extra columns stay out of the scorer.
    frame = pd.read_csv(path, usecols=features, dtype="float64")[features]
    if frame.empty or not np.isfinite(frame.to_numpy()).all():
        raise ValueError("Expected at least one row with finite numeric values in every input.")
    frame.index = pd.RangeIndex(1, len(frame) + 1, name="source_record")
    if replay:
        frame = frame.sort_values("Time", kind="stable")
        first_test_time = frame.Time.iloc[int(.85 * len(frame))]
        frame = frame[frame.Time >= first_test_time]
    return frame

replay = source == "Held-out replay"
if not replay and not custom_csv.strip():
    raise ValueError("Enter the path to your custom CSV in the first cell.")
csv_path = project / "Dataset" / "creditcard.csv" if replay else Path(custom_csv).expanduser()
batch = read_batch(csv_path, replay)
print(f"Loaded {len(batch):,} records from {csv_path.name}.")

Loaded 284,807 records from creditcard.csv.


In [14]:
@torch.inference_mode()
def score_batch(frame, threshold):
    # Reuse training means and scales. Fitting on the incoming batch would change its scores.
    values = frame[features].to_numpy(dtype=np.float64)
    scaled = ((values - np.asarray(saved["mean"])) / np.asarray(saved["scale"])).astype("float32")
    if not np.isfinite(scaled).all():
        raise ValueError("Scaled inputs exceed the range supported by the ANN.")
    chunks = torch.from_numpy(scaled).split(4096)
    scores = torch.cat([model(chunk).flatten().sigmoid() for chunk in chunks]).numpy()
    result = frame[["Time", "Amount"]].copy()
    result["risk_score"] = scores.astype("float64")
    result["send_for_review"] = scores >= threshold
    result["model_run"] = run_id
    result["threshold"] = threshold
    return result.reset_index()

scored = score_batch(batch, cutoff)
queue = scored.loc[scored.send_for_review].sort_values(
    ["risk_score", "source_record"], ascending=[False, True])
print(f"{len(queue):,} alerts from {len(scored):,} records ({len(queue)/len(scored):.2%}).")
display(queue.head(10).style.format({"risk_score": "{:.9f}", "threshold": "{:.9f}"}))

434 alerts from 284,807 records (0.15%).


,source_record,Time,Amount,risk_score,send_for_review,model_run,threshold
6108,6109,6986.000000,59.000000,1.000000000,True,20261121T162317186996Z,0.999876618
6331,6332,7526.000000,1.000000,1.000000000,True,20261121T162317186996Z,0.999876618
6334,6335,7535.000000,1.000000,1.000000000,True,20261121T162317186996Z,0.999876618
6446,6447,7672.000000,1.000000,1.000000000,True,20261121T162317186996Z,0.999876618
6717,6718,8408.000000,1.000000,1.000000000,True,20261121T162317186996Z,0.999876618
6719,6720,8415.000000,1.000000,1.000000000,True,20261121T162317186996Z,0.999876618
6734,6735,8451.000000,1.000000,1.000000000,True,20261121T162317186996Z,0.999876618
6774,6775,8528.000000,1.000000,1.000000000,True,20261121T162317186996Z,0.999876618
6820,6821,8614.000000,1.000000,1.000000000,True,20261121T162317186996Z,0.999876618
6870,6871,8757.000000,1.000000,1.000000000,True,20261121T162317186996Z,0.999876618


## The two CSVs

`scored_transactions.csv` accounts for every input row. `review_queue.csv` contains the flags, highest score first. The reference replay produced 39 alerts from 42,724 transactions. Custom batches have their own output folder.

In [15]:
parent = project / "Artifacts" / "Batch runs"
parent.mkdir(parents=True, exist_ok=True)
mode = "heldout" if replay else "custom"
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
destination = parent / f"{mode}_{stamp}_{uuid4().hex[:8]}"
# Publish the run folder after both CSVs have been written.
with TemporaryDirectory(dir=parent) as temp:
    staging = Path(temp) / "results"
    staging.mkdir()
    scored.to_csv(staging / "scored_transactions.csv", index=False)
    queue.to_csv(staging / "review_queue.csv", index=False)
    staging.rename(destination)
print(destination)

/content/drive/MyDrive/Colab Notebooks/Project 01: Intelligent Financial Fraud Detection/Artifacts/Batch runs/custom_20261121T181912Z_5980d07d
